# GPU Scaling Analysis for Roman Disperser

This notebook analyzes GPU scaling benchmarks for the Roman disperser module.

**Benchmarks cover:**
- Galaxy count scaling: 100, 250, 500, 1000 galaxies
- All three spectral orders: +1, 0, +2
- Wavelength chunk sizes: 50, 100, 200

**Galaxy configuration:**
- 50×50 native pixels (150×150 with 3× oversampling)
- Exponential profile (Sersic n=1)
- Sloped spectrum with edge roll-off (1000 wavelength samples, 1.0-2.0 μm)
- Random positions across full detector (0-4088 in x and y)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread

# Presentation-quality settings
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# Color palette for orders
ORDER_COLORS = {'1': '#1f77b4', '0': '#ff7f0e', '2': '#2ca02c'}
ORDER_LABELS = {'1': 'Order +1', '0': 'Order 0', '2': 'Order +2'}

# Color palette for chunk sizes
CHUNK_COLORS = {50: '#d62728', 100: '#9467bd', 200: '#8c564b'}

## Load Benchmark Results

In [ ]:
# Load benchmark results
results_path = Path("../../scripts/output/benchmark_results.json")

with open(results_path) as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data['results'])

print(f"Loaded {len(df)} benchmark results")
print(f"\nGalaxy counts: {sorted(df['n_galaxies'].unique())}")
print(f"Orders: {sorted(df['order'].unique())}")
print(f"Chunk sizes: {sorted(df['chunk_size'].unique())}")

## GPU Hardware Information

In [ ]:
metadata = data['metadata']
gpu = metadata['gpu']
config = metadata['config']

print("=" * 60)
print("GPU HARDWARE")
print("=" * 60)
print(f"  GPU Name:        {gpu['name']}")
print(f"  Memory:          {gpu['memory_total_gb']:.1f} GB")
print(f"  CUDA Version:    {gpu['cuda_version']}")
print(f"  Driver Version:  {gpu['driver_version']}")
print(f"\n  JAX Backend:     {metadata['jax_backend']}")
print(f"  JAX Devices:     {metadata['jax_devices']}")

print("\n" + "=" * 60)
print("BENCHMARK CONFIGURATION")
print("=" * 60)
print(f"  SCA:             {config['sca']}")
print(f"  Galaxy size:     {config['npix_native']}×{config['npix_native']} native pixels")
print(f"  Oversampling:    {config['oversample']}× ({config['npix_native']*config['oversample']}×{config['npix_native']*config['oversample']} input)")
print(f"  Wavelengths:     {config['n_wavelength']} samples")
print(f"  λ range:         {config['lam_range'][0]}-{config['lam_range'][1]} μm")
print(f"  Position range:  X={config['x_range']}, Y={config['y_range']}")
print(f"  Warmup runs:     {config['n_warmup']}")
print(f"  Timed runs:      {config['n_runs']}")
print(f"\n  Timestamp:       {metadata['timestamp']}")

## Dispersed Detector Image

Combined image of 1000 galaxies dispersed through all three spectral orders (+1, 0, +2) and accumulated onto a single detector.

In [ ]:
# Load and display the combined detector image
image_path = Path("../../scripts/output/dispersed_1000_galaxies.png")

fig, ax = plt.subplots(figsize=(12, 12))
img = imread(image_path)
ax.imshow(img)
ax.axis('off')
ax.set_title('1000 Galaxies - All Orders Combined', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Galaxy Count Scaling

How does dispersion time scale with the number of galaxies? Each panel shows a different spectral order.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

orders = ['1', '0', '2']
chunk_sizes = sorted(df['chunk_size'].unique())

for idx, order in enumerate(orders):
    ax = axes[idx]
    order_df = df[df['order'] == order]
    
    for chunk_size in chunk_sizes:
        subset = order_df[order_df['chunk_size'] == chunk_size].sort_values('n_galaxies')
        ax.errorbar(
            subset['n_galaxies'], 
            subset['mean_time_s'],
            yerr=subset['std_time_s'],
            marker='o',
            markersize=8,
            linewidth=2,
            capsize=4,
            capthick=2,
            color=CHUNK_COLORS[chunk_size],
            label=f'chunk={chunk_size}',
        )
    
    ax.set_xlabel('Number of Galaxies')
    ax.set_ylabel('Time (seconds)')
    ax.set_title(ORDER_LABELS[order], fontweight='bold', color=ORDER_COLORS[order])
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1100)

plt.suptitle('Dispersion Time vs Galaxy Count by Spectral Order', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Scaling Efficiency (Log-Log)

A log-log plot helps verify linear scaling. A slope of 1 indicates perfect linear scaling with galaxy count.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Use chunk_size=100 as reference
chunk_ref = 100
subset = df[df['chunk_size'] == chunk_ref]

for order in orders:
    order_data = subset[subset['order'] == order].sort_values('n_galaxies')
    ax.loglog(
        order_data['n_galaxies'], 
        order_data['mean_time_s'],
        marker='o',
        markersize=10,
        linewidth=2.5,
        color=ORDER_COLORS[order],
        label=ORDER_LABELS[order],
    )

# Add reference line for linear scaling
n_gals = np.array([100, 1000])
# Fit line to order 1 data for reference
order1_data = subset[subset['order'] == '1'].sort_values('n_galaxies')
t_100 = order1_data[order1_data['n_galaxies'] == 100]['mean_time_s'].values[0]
linear_ref = t_100 * (n_gals / 100)
ax.loglog(n_gals, linear_ref, 'k--', linewidth=1.5, alpha=0.5, label='Linear scaling (slope=1)')

ax.set_xlabel('Number of Galaxies')
ax.set_ylabel('Time (seconds)')
ax.set_title(f'Scaling Efficiency (chunk_size={chunk_ref})', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## Time per Galaxy Analysis

How does the per-galaxy overhead change with batch size? Lower is better.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Use chunk_size=100
subset = df[df['chunk_size'] == 100]

# Prepare data for grouped bar chart
n_galaxies_list = sorted(subset['n_galaxies'].unique())
x = np.arange(len(n_galaxies_list))
width = 0.25

for i, order in enumerate(orders):
    order_data = subset[subset['order'] == order].sort_values('n_galaxies')
    values = order_data['time_per_galaxy_ms'].values
    bars = ax.bar(
        x + (i - 1) * width,
        values,
        width,
        label=ORDER_LABELS[order],
        color=ORDER_COLORS[order],
        edgecolor='white',
        linewidth=1,
    )
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.annotate(
            f'{val:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=9,
        )

ax.set_xlabel('Number of Galaxies')
ax.set_ylabel('Time per Galaxy (ms)')
ax.set_title('Per-Galaxy Processing Time by Batch Size and Order (chunk_size=100)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(n_galaxies_list)
ax.legend(loc='upper right')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Chunk Size Comparison

How does the wavelength chunk size affect performance? This parameter controls memory vs. computation tradeoff.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Absolute time vs Chunk Size at max galaxy count
n_fixed = df['n_galaxies'].max()
subset = df[df['n_galaxies'] == n_fixed]

ax = axes[0]
for order in orders:
    order_data = subset[subset['order'] == order].sort_values('chunk_size')
    ax.plot(
        order_data['chunk_size'], 
        order_data['mean_time_s'], 
        marker='o',
        markersize=10,
        linewidth=2.5,
        color=ORDER_COLORS[order],
        label=ORDER_LABELS[order],
    )
ax.set_xlabel('Wavelength Chunk Size')
ax.set_ylabel('Time (seconds)')
ax.set_title(f'Absolute Time (n={n_fixed} galaxies)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(chunk_sizes)

# Right: Relative performance (normalized to chunk=100)
ax = axes[1]
for order in orders:
    order_data = subset[subset['order'] == order].sort_values('chunk_size')
    baseline = order_data[order_data['chunk_size'] == 100]['mean_time_s'].values[0]
    relative = order_data['mean_time_s'] / baseline * 100
    ax.plot(
        order_data['chunk_size'], 
        relative,
        marker='o',
        markersize=10,
        linewidth=2.5,
        color=ORDER_COLORS[order],
        label=ORDER_LABELS[order],
    )

ax.axhline(100, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Wavelength Chunk Size')
ax.set_ylabel('Relative Time (% of chunk=100)')
ax.set_title('Chunk Size Efficiency', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(chunk_sizes)

plt.tight_layout()
plt.show()

## Memory Analysis

Peak GPU memory usage during dispersion. The dashed line shows the GPU's total available memory.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Use chunk_size=100 as reference
subset = df[df['chunk_size'] == 100]

for order in orders:
    order_df = subset[subset['order'] == order].sort_values('n_galaxies')
    memory_gb = order_df['peak_memory_bytes'] / 1024**3
    ax.plot(
        order_df['n_galaxies'], 
        memory_gb.values, 
        marker='o',
        markersize=10,
        linewidth=2.5,
        color=ORDER_COLORS[order],
        label=ORDER_LABELS[order],
    )

# Add GPU memory limit
gpu_memory = gpu['memory_total_gb']
ax.axhline(gpu_memory, color='red', linestyle='--', linewidth=2, 
           label=f'GPU Memory ({gpu_memory:.0f} GB)', alpha=0.7)

ax.set_xlabel('Number of Galaxies')
ax.set_ylabel('Peak Memory (GB)')
ax.set_title('GPU Memory Usage vs Galaxy Count (chunk_size=100)', fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1100)
ax.set_ylim(0, gpu_memory * 1.1)

plt.tight_layout()
plt.show()

## Summary Table

In [ ]:
# Create summary table (chunk_size=100 only for clarity)
summary_df = df[df['chunk_size'] == 100].copy()
summary_df['Peak Memory (GB)'] = summary_df['peak_memory_bytes'] / 1024**3

# Pivot for clean display
pivot = summary_df.pivot_table(
    index='n_galaxies',
    columns='order',
    values=['mean_time_s', 'time_per_galaxy_ms', 'Peak Memory (GB)'],
)

# Round for display
pivot = pivot.round(3)

print("=" * 80)
print("SUMMARY STATISTICS (chunk_size=100)")
print("=" * 80)
print("\nMean Time (seconds):")
print(pivot['mean_time_s'].to_string())
print("\nTime per Galaxy (ms):")
print(pivot['time_per_galaxy_ms'].to_string())
print("\nPeak Memory (GB):")
print(pivot['Peak Memory (GB)'].to_string())

## Multi-Order Timing

Total time to process all three spectral orders for each galaxy count.

In [ ]:
print("=" * 70)
print("MULTI-ORDER ANALYSIS (all 3 orders combined, chunk_size=100)")
print("=" * 70)
print(f"\n{'N_galaxies':>12} {'Total Time':>14} {'Per Galaxy':>14} {'Throughput':>16}")
print("-" * 70)

subset = df[df['chunk_size'] == 100]

for n_gal in sorted(df['n_galaxies'].unique()):
    gal_data = subset[subset['n_galaxies'] == n_gal]
    total_time = gal_data['mean_time_s'].sum()
    per_galaxy_ms = total_time / n_gal * 1000
    throughput = n_gal / total_time
    print(f"{n_gal:>12} {total_time:>12.2f} s {per_galaxy_ms:>12.2f} ms {throughput:>12.1f} gal/s")

print("-" * 70)

## Key Findings

In [ ]:
# Calculate key metrics
subset_100 = df[df['chunk_size'] == 100]
subset_1000 = subset_100[subset_100['n_galaxies'] == 1000]

total_time_1000 = subset_1000['mean_time_s'].sum()
per_galaxy_1000 = total_time_1000 / 1000 * 1000  # ms
throughput_1000 = 1000 / total_time_1000

# Order 0 is typically slowest
order0_time = subset_1000[subset_1000['order'] == '0']['mean_time_s'].values[0]
order1_time = subset_1000[subset_1000['order'] == '1']['mean_time_s'].values[0]
order2_time = subset_1000[subset_1000['order'] == '2']['mean_time_s'].values[0]

# Memory usage
max_memory_gb = df['peak_memory_bytes'].max() / 1024**3
memory_percent = max_memory_gb / gpu['memory_total_gb'] * 100

print("=" * 70)
print("KEY FINDINGS")
print("=" * 70)
print(f"""
1. PERFORMANCE (1000 galaxies, all 3 orders):
   - Total time: {total_time_1000:.2f} seconds
   - Per galaxy: {per_galaxy_1000:.2f} ms
   - Throughput: {throughput_1000:.1f} galaxies/second

2. ORDER COMPARISON (1000 galaxies):
   - Order +1: {order1_time:.2f}s ({order1_time/total_time_1000*100:.1f}% of total)
   - Order  0: {order0_time:.2f}s ({order0_time/total_time_1000*100:.1f}% of total)
   - Order +2: {order2_time:.2f}s ({order2_time/total_time_1000*100:.1f}% of total)

3. MEMORY USAGE:
   - Peak: {max_memory_gb:.2f} GB ({memory_percent:.1f}% of {gpu['memory_total_gb']:.0f} GB available)
   - Memory-efficient chunking keeps usage well under GPU limit

4. SCALING:
   - Linear scaling with galaxy count (as expected)
   - Chunk size has minimal impact on timing
   - Order 0 is ~3x slower than orders +1 and +2 (less dispersion = more overlap)

5. GPU: {gpu['name']}
""")